# Mars Rover Photos
## Deliverable 3: Collect, Analyze and Visualize the Latest Rover Images

This notebook pulls the newest raw images from NASA's Perseverance and Curiosity rovers, analyzes them with Pandas,
and builds a live dashboard (`rover_dashboard.html`) you can open in any browser.

> **Note on the data source:** the old `api.nasa.gov/mars-photos` API was archived in October 2025.
> This notebook uses the JSON feeds behind NASA's own raw-image galleries instead. They need no API key.

In [ ]:
# Import relevant libraries
import json
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests

### Step 1: Configure the Feeds

Each rover has its own feed with slightly different field names, so each gets its own URL builder and a function that
normalizes a raw item into the same set of columns:

* `id`: NASA's image identifier
* `sol`: the Martian day the image was taken, counted from landing
* `utc`: when the image was taken, in Earth UTC time
* `camera`: the instrument code, such as `NAVCAM_LEFT`
* `thumb`, `img`, `full`: small, large and full-resolution image URLs
* `link`: the image's page on the NASA website

In [ ]:
PAGE_SIZE = 100  # photos per request
PAGES = 3        # requests per rover

HEADERS = {"User-Agent": "Mars-Web_scraping notebook"}


def perseverance_url(page):
    return ("https://mars.nasa.gov/rss/api/?feed=raw_images&category=mars2020&feedtype=json"
            f"&num={PAGE_SIZE}&page={page}&order=sol+desc&extended=sample_type::full,")


def curiosity_url(page):
    return ("https://mars.nasa.gov/api/v1/raw_image_items/?order=sol%20desc,date_taken%20desc"
            f"&per_page={PAGE_SIZE}&page={page}&condition_1=msl:mission&extended=thumbnail::sample_type::noteq")


def norm_perseverance(item):
    files = item.get("image_files") or {}
    return {
        "id": item.get("imageid"),
        "sol": item.get("sol"),
        "utc": item.get("date_taken_utc"),
        "camera": (item.get("camera") or {}).get("instrument", "UNKNOWN"),
        "thumb": files.get("small") or files.get("medium") or files.get("large"),
        "img": files.get("large") or files.get("medium") or files.get("full_res"),
        "full": files.get("full_res") or files.get("large"),
        "link": item.get("link", ""),
    }


def norm_curiosity(item):
    url = item.get("https_url") or item.get("url")
    return {
        "id": item.get("imageid"),
        "sol": item.get("sol"),
        "utc": item.get("date_taken"),
        "camera": item.get("instrument", "UNKNOWN"),
        "thumb": url,
        "img": url,
        "full": url,
        "link": item.get("link", ""),
    }


FEEDS = {
    "perseverance": {"url": perseverance_url, "key": "images", "norm": norm_perseverance},
    "curiosity": {"url": curiosity_url, "key": "items", "norm": norm_curiosity},
}

### Step 2: Fetch the Latest Photos

Request a few pages from each feed, newest sol first, and collect the normalized rows.

In [ ]:
def fetch_rover(rover):
    feed = FEEDS[rover]
    rows = []
    for page in range(PAGES):
        response = requests.get(feed["url"](page), headers=HEADERS, timeout=30)
        response.raise_for_status()
        items = response.json().get(feed["key"], [])
        rows.extend(feed["norm"](item) for item in items)
        if len(items) < PAGE_SIZE:
            break
    return rows


raw = {}
for rover in FEEDS:
    try:
        raw[rover] = fetch_rover(rover)
        print(f"{rover}: {len(raw[rover])} photos")
    except requests.RequestException as err:
        raw[rover] = []
        print(f"{rover}: request failed ({err})")

### Step 3: Store the Data

Combine both rovers into one DataFrame and cast each column to the right type.

In [ ]:
df = pd.concat(
    [pd.DataFrame(rows).assign(rover=rover) for rover, rows in raw.items() if rows],
    ignore_index=True,
)
df = df.dropna(subset=["id", "sol", "thumb"]).drop_duplicates(subset=["rover", "id"])
df["sol"] = df["sol"].astype(int)
# The two feeds format timestamps differently, so parse each value on its own
df["utc"] = pd.to_datetime(df["utc"].map(lambda s: pd.to_datetime(s, utc=True, errors="coerce")), utc=True)
df.head()

In [ ]:
# Examine data type of each column
df.dtypes

### Step 4: Analyze the Data

Answer a few questions about what each rover has been photographing:

1. What is the latest sol each rover has sent images from?
2. How many photos came down on each sol?
3. Which cameras took the most photos?

In [ ]:
# 1. Latest sol and newest capture time for each rover
summary = df.groupby("rover").agg(
    latest_sol=("sol", "max"),
    newest_capture_utc=("utc", "max"),
    photos=("id", "count"),
    sols=("sol", "nunique"),
    cameras=("camera", "nunique"),
)
summary

In [ ]:
# 2. Photos per sol for each rover
photos_per_sol = df.groupby(["rover", "sol"]).size().rename("photos").reset_index()
photos_per_sol.sort_values(["rover", "sol"], ascending=[True, False]).groupby("rover").head(10)

In [ ]:
# 3. Photos by camera for each rover
photos_per_camera = df.groupby(["rover", "camera"]).size().rename("photos").reset_index()
photos_per_camera.sort_values(["rover", "photos"], ascending=[True, False])

In [ ]:
# Plot the photos by camera for each rover
rovers = list(df["rover"].unique())
fig, axes = plt.subplots(1, len(rovers), figsize=(7 * len(rovers), 5), squeeze=False)
for ax, rover in zip(axes[0], rovers):
    counts = (photos_per_camera[photos_per_camera["rover"] == rover]
              .sort_values("photos").tail(10))
    ax.barh(counts["camera"], counts["photos"], color="#b0472a")
    ax.set_title(f"{rover.title()}: photos by camera")
    ax.set_xlabel("Number of photos")
plt.tight_layout()
plt.show()

### Step 5: Save the Data

Save the photo list to a CSV file.

In [ ]:
df.to_csv("rover_photos.csv", index=False)

### Step 6: Build the Live Dashboard

Write the photos into `dashboard_template.html` as a snapshot and save the result as `rover_dashboard.html`.
Open that file in a browser. It shows the snapshot right away, then tries to refresh straight from NASA
and keeps refreshing every 10 minutes. If the browser blocks the direct request, the snapshot stays on screen;
re-run this notebook to update it.

In [ ]:
snapshot = {
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "rovers": {},
}
for rover, group in df.groupby("rover"):
    records = group.assign(utc=group["utc"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")).drop(columns="rover")
    snapshot["rovers"][rover] = records.to_dict(orient="records")

template = Path("dashboard_template.html").read_text(encoding="utf-8")
marker = "/*__SNAPSHOT__*/null"
assert marker in template, "Snapshot marker not found in dashboard_template.html"
data = json.dumps(snapshot, separators=(",", ":")).replace("</", "<\\/")
Path("rover_dashboard.html").write_text(template.replace(marker, data), encoding="utf-8")
print(f"Wrote rover_dashboard.html with {len(df)} photos")